# ETA por distancia y riesgo de retraso — versión 2

Esta revisión añade distancia origen–destino a la predicción y entrena otro modelo para estimar la probabilidad de superar un plazo prometido. Se preserva la versión temporal anterior. Los resultados se leen de evidencia real; las consultas de ejemplo son simuladas.

El [informe oficial de Meituan, sección 2.1](https://github.com/meituan/Meituan-INFORMS-TSL-Research-Challenge/blob/main/tsl_meituan_2024_data_report_20241015.pdf) permite usar distancias relativas con coordenadas desplazadas. El divisor 1e6 se interpreta como codificación en microgrados; esta conversión es una inferencia documentada, no una cita literal del informe. No hay validación para El Salvador.


In [ ]:
from pathlib import Path
import json
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
BUNDLE = ROOT / 'models/delivery-duration/meituan-eta-risk-v2'
if not BUNDLE.exists():
    raise RuntimeError('Se requiere el bundle propio v2; ver comando de entrenamiento en README. En Colab preparar antes la fuente real con notebook 01 y usar el ZIP de fuentes actualizado.')
metadata = json.loads((BUNDLE / 'metadata.json').read_text())
evidence = json.loads((BUNDLE / 'evidence.json').read_text())
evidence['rows']


## Comparación real y selección

La selección usa tres folds temporales y sus etiquetas disponibles. El test es el mismo de la versión 1, ya observado: es una revisión exploratoria. No se elige el ganador por el menor error de test. Los candidatos de duración son mediana, regresión lineal, Random Forest y gradient boosting; riesgo compara prevalencia, regresión logística y gradient boosting.


In [ ]:
tables = {}
for task, experiment in evidence['experiments'].items():
    rows = []
    for name, result in experiment['candidates'].items():
        rows.append({'algorithm': name, 'cv_score': result['mean_score'],
                     'selected_by_cv': name == experiment['selected_by_cv'], **result['test']})
    tables[task] = pd.DataFrame(rows).set_index('algorithm')
display(tables['eta'])
display(tables['risk'])


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
tables['eta']['mae_minutes'].plot.bar(ax=axes[0], title='Error absoluto medio — minutos')
tables['risk']['roc_auc'].plot.bar(ax=axes[1], title='Discriminación de retrasos — ROC AUC')
axes[1].set_ylim(0, 1)
plt.tight_layout()
plt.show()


## Consultar ambos modelos

Misma fecha y distinto recorrido/plazo. Las duraciones y probabilidades siguientes sí proceden de los modelos entrenados; las coordenadas y plazos son escenarios ilustrativos. La tarifa, asignación y ROI siguen siendo reglas del simulador, no etiquetas inventadas de ML.


In [ ]:
from datetime import datetime, timezone
from delivery.domain.models import Location
from delivery_ml.delivery_v2 import DeliveryPredictorV2
model = DeliveryPredictorV2(BUNDLE)
scenarios = []
for longitude, promise in [(0.01, 30), (0.05, 30), (0.05, 45)]:
    scenarios.append(model.predict(datetime(2022, 10, 22, 18, tzinfo=timezone.utc),
                                   Location(0, 0), Location(0, longitude), promise))
pd.DataFrame(scenarios)[['distance_km', 'promised_minutes', 'estimated_minutes', 'late_probability', 'outside_training_range']]


## Usarlo en la aplicación

`POST /predict/delivery` recibe fecha, origen, destino y plazo prometido. La demo completa de `scripts/demo_delivery.py` usa esos dos modelos y conserva el resultado en `data/reports/presentation/demo_result.json`.

Para reproducir el entrenamiento con la fuente real: `OMP_NUM_THREADS=2 OPENBLAS_NUM_THREADS=2 uv run --locked python ml/experiments/train_delivery_v2.py --data-root data --output models/delivery-duration/meituan-eta-risk-v2`. El directorio de salida debe ser nuevo; no se sobrescriben resultados. En Colab se necesita la fuente de código actualizada y los datos reales preparados con el notebook 01. Este notebook no descarga registros ni inicia entrenamiento automáticamente.
